In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
from numpy.linalg import norm
import os
import pandas as pd

from scipy.stats import skew
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

In [2]:
input_directory  = '/eos/home-i00/d/dhnaik/C2V_event_training_data'
with open('/eos/home-i00/d/dhnaik/C2V_event_training_data/meta_dict.json') as f:
    meta_dict = json.load(f)
feature_dict = meta_dict['input_vars']
class_labels = meta_dict['class_labels']

In [3]:
print(list(feature_dict.keys()))

['L1T_JetPuppiAK4_PT0', 'L1T_JetPuppiAK4_PT1', 'L1T_JetPuppiAK4_PT2', 'L1T_JetPuppiAK4_PT3', 'L1T_JetPuppiAK4_PT4', 'L1T_JetPuppiAK4_PT5', 'L1T_JetPuppiAK4_PT6', 'L1T_JetPuppiAK4_PT7', 'L1T_JetPuppiAK4_PT8', 'L1T_JetPuppiAK4_PT9', 'L1T_JetPuppiAK4_Eta0', 'L1T_JetPuppiAK4_Eta1', 'L1T_JetPuppiAK4_Eta2', 'L1T_JetPuppiAK4_Eta3', 'L1T_JetPuppiAK4_Eta4', 'L1T_JetPuppiAK4_Eta5', 'L1T_JetPuppiAK4_Eta6', 'L1T_JetPuppiAK4_Eta7', 'L1T_JetPuppiAK4_Eta8', 'L1T_JetPuppiAK4_Eta9', 'L1T_JetPuppiAK4_Phi0', 'L1T_JetPuppiAK4_Phi1', 'L1T_JetPuppiAK4_Phi2', 'L1T_JetPuppiAK4_Phi3', 'L1T_JetPuppiAK4_Phi4', 'L1T_JetPuppiAK4_Phi5', 'L1T_JetPuppiAK4_Phi6', 'L1T_JetPuppiAK4_Phi7', 'L1T_JetPuppiAK4_Phi8', 'L1T_JetPuppiAK4_Phi9', 'L1T_MuonTight_PT0', 'L1T_MuonTight_PT1', 'L1T_MuonTight_PT2', 'L1T_MuonTight_PT3', 'L1T_MuonTight_Eta0', 'L1T_MuonTight_Eta1', 'L1T_MuonTight_Eta2', 'L1T_MuonTight_Eta3', 'L1T_MuonTight_Phi0', 'L1T_MuonTight_Phi1', 'L1T_MuonTight_Phi2', 'L1T_MuonTight_Phi3', 'L1T_Electron_PT0', 'L1T_Elec

In [4]:
print(f'class labels: {class_labels}')

class labels: {'QCD': 0, 'HH_4b': 1}


In [5]:
X_features = np.load(input_directory+'/train/X_features.npy')
y_labels   = np.load(input_directory+'/train/y_label.npy')

In [6]:
def preprocess(X,y,create_and_slice_features, normalise=False, test_size=0.3):
    
    X = create_and_slice_features(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y, shuffle=True)
    
    if normalise:
    
        train_mask = (X_train[..., 0] != 0)[..., np.newaxis] 
        test_mask = (X_test[..., 0] != 0)[..., np.newaxis]
        
        mean = np.mean(X_train, axis=(0, 1), keepdims=True)
        std = np.std(X_train, axis=(0, 1), keepdims=True)
        
        std = np.where(std == 0, 1e-7, std)
    
        # Normalize
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std
        
        # RE-MASK: Force the padded particles back to exactly 0.0
        X_train = X_train * train_mask
        X_test = X_test * test_mask

        
    return X_train, X_test, y_train, y_test

In [7]:
def just_feature_vector(X_cand,features=[],features_list=[]):
    selected_features = [features_list[feature] for feature in features ]
    X = X_cand[:,selected_features]
    
    return X

In [8]:
jet_feature_list = ['L1T_JetPuppiAK4_PT','L1T_JetPuppiAK4_Eta','L1T_JetPuppiAK4_Phi']
muon_feature_list = ['L1T_MuonTight_PT','L1T_MuonTight_Eta','L1T_MuonTight_Phi']
electron_feature_list = ['L1T_Electron_PT','L1T_Electron_Eta','L1T_Electron_Phi']
met_feature_list = ['L1T_PUPPIMET_MET','L1T_PUPPIMET_Eta','L1T_PUPPIMET_Phi']

max_number_of_jets = 10
max_number_of_muons = 4
max_number_of_electrons = 4

top_x_jets = [feature + str(i) for i in range(max_number_of_jets) for feature in jet_feature_list ]
top_x_muons = [feature + str(i) for i in range(max_number_of_muons) for feature in muon_feature_list]
top_x_electrons = [feature + str(i) for i in range(max_number_of_electrons) for feature in electron_feature_list]
all_columns = top_x_jets + top_x_muons + top_x_electrons + met_feature_list

In [9]:
X_train, X_test, y_train, y_test = preprocess(X_features,y_labels, 
                                              lambda X_features: just_feature_vector(X_features,features=all_columns,features_list=feature_dict),
                                              normalise=False)

In [10]:
model_input_shape = X_train.shape[1:]
model_output_shape = len(class_labels.keys())
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print(f'Feature Names: {all_columns}')
print(f'\nFeature Names: {list(feature_dict.keys())}')

Train Shape: (1259798, 57) | Test Shape: (539914, 57)
Feature Names: ['L1T_JetPuppiAK4_PT0', 'L1T_JetPuppiAK4_Eta0', 'L1T_JetPuppiAK4_Phi0', 'L1T_JetPuppiAK4_PT1', 'L1T_JetPuppiAK4_Eta1', 'L1T_JetPuppiAK4_Phi1', 'L1T_JetPuppiAK4_PT2', 'L1T_JetPuppiAK4_Eta2', 'L1T_JetPuppiAK4_Phi2', 'L1T_JetPuppiAK4_PT3', 'L1T_JetPuppiAK4_Eta3', 'L1T_JetPuppiAK4_Phi3', 'L1T_JetPuppiAK4_PT4', 'L1T_JetPuppiAK4_Eta4', 'L1T_JetPuppiAK4_Phi4', 'L1T_JetPuppiAK4_PT5', 'L1T_JetPuppiAK4_Eta5', 'L1T_JetPuppiAK4_Phi5', 'L1T_JetPuppiAK4_PT6', 'L1T_JetPuppiAK4_Eta6', 'L1T_JetPuppiAK4_Phi6', 'L1T_JetPuppiAK4_PT7', 'L1T_JetPuppiAK4_Eta7', 'L1T_JetPuppiAK4_Phi7', 'L1T_JetPuppiAK4_PT8', 'L1T_JetPuppiAK4_Eta8', 'L1T_JetPuppiAK4_Phi8', 'L1T_JetPuppiAK4_PT9', 'L1T_JetPuppiAK4_Eta9', 'L1T_JetPuppiAK4_Phi9', 'L1T_MuonTight_PT0', 'L1T_MuonTight_Eta0', 'L1T_MuonTight_Phi0', 'L1T_MuonTight_PT1', 'L1T_MuonTight_Eta1', 'L1T_MuonTight_Phi1', 'L1T_MuonTight_PT2', 'L1T_MuonTight_Eta2', 'L1T_MuonTight_Phi2', 'L1T_MuonTight_PT3', 'L1T

## `yggdrasil gradient boosted decision tree`

In [14]:
import ydf

In [13]:
event_train_ds = {f'{all_columns[i]}' : X_train[:,i] for i in range(X_train.shape[1])}
event_train_ds['label'] = y_train.astype(int)

event_test_ds = {f'{all_columns[i]}' : X_test[:,i] for i in range(X_test.shape[1])}
event_test_ds['label'] = y_test.astype(int)

event_train_ds = pd.DataFrame(event_train_ds)
event_test_ds = pd.DataFrame(event_test_ds)

In [15]:
event_train_ds

,L1T_JetPuppiAK4_PT0,L1T_JetPuppiAK4_Eta0,L1T_JetPuppiAK4_Phi0,L1T_JetPuppiAK4_PT1,L1T_JetPuppiAK4_Eta1,L1T_JetPuppiAK4_Phi1,L1T_JetPuppiAK4_PT2,L1T_JetPuppiAK4_Eta2,L1T_JetPuppiAK4_Phi2,L1T_JetPuppiAK4_PT3,...,L1T_Electron_PT2,L1T_Electron_Eta2,L1T_Electron_Phi2,L1T_Electron_PT3,L1T_Electron_Eta3,L1T_Electron_Phi3,L1T_PUPPIMET_MET,L1T_PUPPIMET_Eta,L1T_PUPPIMET_Phi,label
0,73.750000,0.000000,0.00000,1.736328,0.000000,0.0,2.650391,0.00000,0.0,72.812500,...,0.000000,0.0,0.0,0.0,0.0,0.0,42.312500,3.255859,0.890137,0
1,57.531250,1.315430,0.00000,2.250000,-2.058594,0.0,1.978516,0.00000,0.0,57.093750,...,0.000000,0.0,0.0,0.0,0.0,0.0,54.562500,-2.687500,-1.416992,1
2,23.437500,0.000000,0.00000,1.344727,0.000000,0.0,0.031097,0.00000,0.0,21.953125,...,0.000000,0.0,0.0,0.0,0.0,0.0,5.765625,-3.855469,-2.697266,0
3,243.375000,0.000000,0.00000,0.463623,0.000000,0.0,-0.386719,0.00000,0.0,166.375000,...,0.000000,0.0,0.0,0.0,0.0,0.0,74.687500,-0.861816,3.054688,0
4,42.062500,0.000000,0.00000,1.275391,0.000000,0.0,0.466797,0.00000,0.0,21.968750,...,0.000000,0.0,0.0,0.0,0.0,0.0,36.531250,0.203735,-2.660156,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259793,28.515625,0.000000,0.00000,-1.665039,0.000000,0.0,-2.158203,0.00000,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,32.718750,1.677734,0.778809,0
1259794,364.250000,0.507324,0.72998,0.251953,-1.334961,0.0,0.697266,50.78125,0.0,324.750000,...,-1.795898,0.0,0.0,0.0,0.0,0.0,193.750000,-0.579590,-1.796875,1
1259795,23.234375,0.000000,0.00000,1.959961,0.000000,0.0,1.464844,0.00000,0.0,17.328125,...,0.000000,0.0,0.0,0.0,0.0,0.0,9.679688,-0.790527,-0.260986,0
1259796,41.875000,0.000000,0.00000,0.366455,0.000000,0.0,-2.197266,0.00000,0.0,16.390625,...,0.000000,0.0,0.0,0.0,0.0,0.0,27.375000,2.673828,1.181641,0


In [16]:
event_train_subset = pd.concat([
    grp.sample(frac=0.3, random_state=42)
    for _, grp in event_train_ds.groupby('label')
]).reset_index(drop=True)

event_test_subset = pd.concat([
    grp.sample(frac=0.3, random_state=42)
    for _, grp in event_test_ds.groupby('label')
]).reset_index(drop=True)

In [18]:
event_classifier_model = ydf.GradientBoostedTreesLearner(
    label='label',
    num_trees=500,
    shrinkage=0.08,
    subsample=0.8,
    min_examples=100,
    max_depth=8,
    growing_strategy='LOCAL',
    early_stopping='LOSS_INCREASE',
    early_stopping_num_trees_look_ahead=50,
    validation_ratio=0.1,
    l2_regularization=0.2,
    num_threads=64,
).train(event_train_ds, verbose=1)

Train model on 1259798 examples
Model trained in 0:04:22.314876


In [21]:
!pwd

/eos/home-i00/d/dhnaik/SDT


In [22]:
prediction = event_classifier_model.predict(event_test_ds)
prediction_2col = np.stack([1 - prediction, prediction], axis=1)  # (N, 2)
np.save('/eos/home-i00/d/dhnaik/SDT/test_outputs/EVENT_C2V/HH_4b_bdt_probs.npy', prediction_2col)

In [23]:
prediction_2col

array([[0.9756109 , 0.02438909],
       [0.8641733 , 0.13582672],
       [0.85838264, 0.14161737],
       ...,
       [0.12948066, 0.87051934],
       [0.94470036, 0.05529961],
       [0.27982968, 0.7201703 ]], shape=(539914, 2), dtype=float32)

In [24]:
evaluation = event_classifier_model.evaluate(event_test_ds, weighted=False)

In [25]:
evaluation

Label \ Pred,0,1
0,204303,35798
1,26322,273491


In [26]:
event_classifier_model.describe()